# 🛠️ TileForge AI - Google Colab GPU Fallback Server

This notebook provides a **free cloud GPU backend** for TileForge AI using Google Colab (NVIDIA T4 / V100 / A100) and **ngrok** for secure tunneling into your local Unity Editor.

### 📋 Instructions:
1. In the top menu, go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Sign up for a free account at [ngrok.com](https://ngrok.com/) to get your AuthToken.
3. Run the cells below sequentially.
4. Copy the generated **Public ngrok URL** (e.g. `https://xxxx.ngrok-free.app`) into the **TileForge Window** inside the Unity Editor!

In [ ]:
# 1. Verify NVIDIA GPU Acceleration
!nvidia-smi

In [ ]:
# 2. Install Required Dependencies
!pip install -q fastapi uvicorn pydantic diffusers transformers accelerate opencv-python-headless pyngrok nest-asyncio

In [ ]:
# 3. Configure ngrok Authentication
from pyngrok import ngrok, conf
import getpass

print("Enter your ngrok Authtoken (get it from https://dashboard.ngrok.com/get-started/your-authtoken):")
NGROK_TOKEN = getpass.getpass("ngrok Authtoken: ")
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
# 4. Initialize SD-Turbo and FastAPI Backend in Colab
import io
import time
import base64
import torch
import cv2
import numpy as np
from PIL import Image
from diffusers import AutoPipelineForText2Image
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
import nest_asyncio
import uvicorn
import threading

print("Loading SD-Turbo Model into GPU memory...")
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    variant="fp16" if device == "cuda" else None
).to(device)

# Warmup inference
_ = pipe("tile texture", num_inference_steps=1, guidance_scale=0.0)
print("✅ SD-Turbo is warmed up and ready!")

# Pydantic Schemas
class Point2D(BaseModel):
    x: float
    y: float

class TileRequest(BaseModel):
    prompt: str
    negative_prompt: str = "blurry, 3d, noisy"
    width: int = 512
    height: int = 512
    steps: int = 2
    guidance_scale: float = 1.5
    seed: int = -1
    seamless: bool = True
    generate_normal_map: bool = True
    normal_strength: float = 2.5
    generate_collider: bool = True
    collider_tolerance: float = 2.0

class TileResponse(BaseModel):
    status: str
    seed: int
    generation_time_ms: float
    color_map_base64: str
    normal_map_base64: Optional[str] = None
    collider_polygons: List[List[Point2D]] = []
    metadata: Dict[str, Any] = {}

# Image Processing Functions
def apply_seamless_tiling(image: Image.Image, blend_ratio: float = 0.15) -> Image.Image:
    img_np = np.array(image).astype(np.float32)
    h, w, c = img_np.shape
    shift_y, shift_x = h // 2, w // 2
    rolled = np.roll(np.roll(img_np, shift_y, axis=0), shift_x, axis=1)
    bw, bh = int(w * blend_ratio), int(h * blend_ratio)
    linear_x = np.linspace(0, 1, bw, endpoint=True).reshape(1, bw, 1)
    left = rolled[:, :bw]
    right = rolled[:, -bw:]
    rolled[:, :bw] = left * linear_x + right * (1 - linear_x)
    rolled[:, -bw:] = rolled[:, :bw]
    linear_y = np.linspace(0, 1, bh, endpoint=True).reshape(bh, 1, 1)
    top = rolled[:bh, :]
    bottom = rolled[-bh:, :]
    rolled[:bh, :] = top * linear_y + bottom * (1 - linear_y)
    rolled[-bh:, :] = rolled[:bh, :]
    unrolled = np.roll(np.roll(rolled, -shift_y, axis=0), -shift_x, axis=1)
    return Image.fromarray(np.clip(unrolled, 0, 255).astype(np.uint8))

def bake_normal_map(image: Image.Image, strength: float = 2.5) -> Image.Image:
    gray = np.array(image.convert("L")).astype(np.float32) / 255.0
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    nx = -sobel_x * strength
    ny = -sobel_y * strength
    nz = np.ones_like(nx)
    norm = np.sqrt(nx**2 + ny**2 + nz**2) + 1e-6
    nx = (nx / norm * 0.5 + 0.5) * 255
    ny = (ny / norm * 0.5 + 0.5) * 255
    nz = (nz / norm * 0.5 + 0.5) * 255
    normal_map = np.stack([nx, ny, nz], axis=2).astype(np.uint8)
    return Image.fromarray(normal_map)

def extract_colliders(image: Image.Image, tolerance: float = 2.0) -> List[List[Point2D]]:
    img_np = np.array(image)
    if img_np.shape[2] == 4:
        mask = (img_np[:, :, 3] > 30).astype(np.uint8) * 255
    else:
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 40, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h, w = mask.shape
    polygons = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 150:
            continue
        approx = cv2.approxPolyDP(cnt, tolerance, True)
        poly = []
        for pt in approx:
            px, py = pt[0][0], pt[0][1]
            poly.append(Point2D(x=round((px / w) - 0.5, 4), y=round(0.5 - (py / h), 4)))
        if len(poly) >= 3:
            polygons.append(poly)
    return polygons

def pil_to_base64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

# FastAPI App Setup
app = FastAPI(title="TileForge AI - Colab Backend")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.is_available(), "device": device}

@app.post("/api/v1/generate-tile", response_model=TileResponse)
def generate_tile(req: TileRequest):
    t0 = time.time()
    seed = req.seed if req.seed >= 0 else int(time.time() * 1000) % 2147483647
    generator = torch.Generator(device=device).manual_seed(seed)
    image = pipe(
        req.prompt,
        negative_prompt=req.negative_prompt,
        num_inference_steps=req.steps,
        guidance_scale=req.guidance_scale,
        width=req.width,
        height=req.height,
        generator=generator
    ).images[0]
    if req.seamless:
        image = apply_seamless_tiling(image)
    color_b64 = pil_to_base64(image)
    normal_b64 = None
    if req.generate_normal_map:
        normal_img = bake_normal_map(image, strength=req.normal_strength)
        normal_b64 = pil_to_base64(normal_img)
    polygons = []
    if req.generate_collider:
        polygons = extract_colliders(image, tolerance=req.collider_tolerance)
    duration_ms = round((time.time() - t0) * 1000.0, 2)
    return TileResponse(
        status="success",
        seed=seed,
        generation_time_ms=duration_ms,
        color_map_base64=color_b64,
        normal_map_base64=normal_b64,
        collider_polygons=polygons,
        metadata={"device": device, "steps": req.steps}
    )

# Start Background Server & Connect ngrok Tunnel
nest_asyncio.apply()
public_url = ngrok.connect(8000)
print("\n" + "="*60)
print(f"🚀 TileForge AI Backend is Live at: {public_url}")
print("="*60)
print("👉 Copy the URL above and paste it into Unity LevelGenWindow!")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
print("Server thread started successfully.")